 # Week 5: Apache Spark - Data Cleaning, Transformation and Aggregation
Objective
Understand Spark fundamentals and perform data cleaning, transformation, and aggregation using DataFrames.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder.appName("Week5_Spark_Assignment").getOrCreate()
# check the session
print("Spark Session Created Successfully")

Spark Session Created Successfully


In [4]:
df = spark.read.csv( "../data/spark_synthetic_dataset.csv", header=True, inferSchema=True )
print("Data set readed")

Data set readed


In [5]:
df.show(5)

+-------+----------------+------+----------------+-----------+------+---+------------+-------+--------+-----------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|  city|age|subscription|  price|store_id|            email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+------+---+------------+-------+--------+-----------------+--------+-------------------+
|   1028|      2026-06-13|  West|        Clothing|    1299.97| Delhi| 22|    Standard| 1494.3|      18|user1@example.com|  user_1|2026-06-13 02:00:00|
|   1108|      2026-06-01|  West|     Electronics|     559.11| Delhi| 48|    Standard| 101.75|       7|user2@example.com|  user_2|2026-06-01 22:00:00|
|   1179|      2026-06-16| South|       Furniture|    2301.12|Mumbai| 16|     Premium|1411.37|      11|             NULL|  user_3|2026-06-16 08:00:00|
|   1055|      2026-02-09| North|     Electronics|     554.45|Jaipur| 38|       Basic|1227.27|

In [7]:
# schema check
df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)



In [12]:
# check rows and columns
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

Total Rows: 1020
Total Columns: 13


In [14]:
# lets look to the statistics
df.describe().show()

+-------+------------------+------+----------------+------------------+---------+------------------+------------+------------------+-----------------+-------------------+--------+
|summary|           user_id|region|product_category|       sale_amount|     city|               age|subscription|             price|         store_id|              email|username|
+-------+------------------+------+----------------+------------------+---------+------------------+------------+------------------+-----------------+-------------------+--------+
|  count|              1020|  1020|            1020|              1020|     1020|              1020|        1020|               980|             1020|                995|     995|
|   mean|1099.8960784313726|  NULL|            NULL|2525.2537843137256|     NULL| 37.77450980392157|        NULL|1005.0782857142856|10.82843137254902|               NULL|    NULL|
| stddev| 59.64907997791512|  NULL|            NULL|1423.7826469363808|     NULL|12.946335118021919|

# Assigments Questions

# Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

# Traditional MapReduce stores intermediate results on disk after every stage, which makes it slower for modern big data workloads.

Limitations of MapReduce:

1. Disk-based processing
2. Slow iterative computations
3. Complex programming model
4. High latency
5. Limited support for advanced analytics

Advantages of Spark:

1. In-memory computing
2. Faster execution (10x–100x)
3. Easy DataFrame APIs
4. Supports SQL, Machine Learning, Streaming and Graph processing

Insight:
Spark reduces disk I/O and efficiently handles iterative workloads.

# Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

# Spark stores reusable data in RAM instead of repeatedly reading it from disk.

Disk-Based System:

Iteration 1 → Read from Disk
Iteration 2 → Read from Disk
Iteration 3 → Read from Disk

Spark In-Memory Processing:

Load Data Once → Store in Memory

Iteration 1 → RAM
Iteration 2 → RAM
Iteration 3 → RAM

Benefits:

- Faster execution
- Reduced disk I/O
- Better performance
- Efficient for Machine Learning algorithms

Insight:
Spark improves performance by keeping reusable datasets in memory.

# Q3: Remove duplicate rows based on user_id and transaction_date

In [16]:
df_no_duplicates = df.dropDuplicates(
    ["user_id", "transaction_date"]
)

print("Rows Before:", df.count())

print("Rows After:", df_no_duplicates.count())

df_no_duplicates.show(5)

Rows Before: 1020
Rows After: 987
+-------+----------------+------+----------------+-----------+---------+---+------------+-------+--------+-------------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|  price|store_id|              email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---------+---+------------+-------+--------+-------------------+--------+-------------------+
|   1000|      2026-01-07|  East|        Clothing|    1570.83|Hyderabad| 56|       Basic| 292.15|      10|user677@example.com|user_677|2026-01-07 13:00:00|
|   1000|      2026-01-11| South|        Clothing|    3299.99|    Delhi| 16|     Premium|1870.34|      16|user814@example.com|user_814|2026-01-11 08:00:00|
|   1000|      2026-02-04| South|       Furniture|    3178.65|   Mumbai| 52|     Premium| 266.11|       8|user733@example.com|user_733|2026-02-04 16:00:00|
|   1000|      2026-04-27| Sou

In [17]:
# Insights There are total the 1020 rows before the execution but after the function working it becomes 987 which will improve the data ouput more best

# Q4: Filter region = 'West' and find the average sale_amount grouped by product_category

In [21]:
west_sales = (
    df.filter(
        col("region") == "West"
    )

    .groupBy(
        "product_category"
    )

    .agg(
        round(avg("sale_amount"),2)
        .alias("average_sale")
    )
    .orderBy(
        col("average_sale").desc()
    )
)

west_sales.show()

+----------------+------------+
|product_category|average_sale|
+----------------+------------+
|        Clothing|     2711.29|
|     Electronics|     2624.43|
|       Groceries|     2447.75|
|       Furniture|      2442.9|
+----------------+------------+



In [22]:
# we can see clearly that the clothing has the highest avg in the west region where as the Furniture has lower

# Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

In [28]:
# .na.drop() this will drop the cells with the null values  and .na.fill() will fill the null values inside the data set with the value

df_status = df.withColumn(
    "status",
    lit(None).cast("string")
)

df_status = df_status.na.fill(
    {"status": "Unknown"}
)

df_status.select("status").show(5)

+-------+
| status|
+-------+
|Unknown|
|Unknown|
|Unknown|
|Unknown|
|Unknown|
+-------+
only showing top 5 rows


# Q6: Find the total count of records for each city where the count is greater than 100.

In [29]:
city_count = (
    df.groupBy("city")
      .count()
      .filter(col("count") > 100)
)

city_count.show()

+---------+-----+
|     city|count|
+---------+-----+
|   Mumbai|  151|
|     Pune|  166|
|    Delhi|  177|
|Bengaluru|  147|
|Hyderabad|  209|
|   Jaipur|  170|
+---------+-----+



# Q7: How does the immutability of Spark DataFrames affect data cleaning?

* Spark DataFrames are immutable.

Operations such as drop(), rename(), or filter() do not modify the original DataFrame. Instead, a new DataFrame is created.

This helps preserve the original data and prevents accidental data loss.

# Q8: Filter rows where age is between 18 and 30 and subscription is 'Premium'.

In [30]:
premium_users = df.filter(
    (col("age").between(18, 30))
    &
    (col("subscription") == "Premium")
)

premium_users.show()

+-------+----------------+------+----------------+-----------+---------+---+------------+-------+--------+-------------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|  price|store_id|              email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---------+---+------------+-------+--------+-------------------+--------+-------------------+
|   1180|      2026-02-19|  West|     Electronics|    3340.19|   Mumbai| 21|     Premium|1739.64|      13|  user6@example.com|  user_6|2026-02-19 08:00:00|
|   1021|      2026-01-21| South|     Electronics|    4892.12|Bengaluru| 24|     Premium|1336.51|      18| user16@example.com| user_16|2026-01-21 05:00:00|
|   1056|      2026-05-31|  West|     Electronics|    3568.42|   Jaipur| 30|     Premium|1815.57|      11| user19@example.com| user_19|2026-05-31 02:00:00|
|   1063|      2026-04-29|  West|       Groceries|    4059.31|Be

# Q9: Why should null values be handled before performing aggregations?

# Null values can lead to inaccurate calculations.

Handling null values before applying sum(), avg(), min(), or max() ensures accurate analysis and better data quality.

# Q10: Cast raw_timestamp to TimestampType and rename it to event_time.

In [31]:
df_timestamp = (
    df.withColumn(
        "event_time",
        col("raw_timestamp").cast(TimestampType())
    )

    .drop("raw_timestamp")
)

df_timestamp.show(5)

+-------+----------------+------+----------------+-----------+------+---+------------+-------+--------+-----------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|  city|age|subscription|  price|store_id|            email|username|         event_time|
+-------+----------------+------+----------------+-----------+------+---+------------+-------+--------+-----------------+--------+-------------------+
|   1028|      2026-06-13|  West|        Clothing|    1299.97| Delhi| 22|    Standard| 1494.3|      18|user1@example.com|  user_1|2026-06-13 02:00:00|
|   1108|      2026-06-01|  West|     Electronics|     559.11| Delhi| 48|    Standard| 101.75|       7|user2@example.com|  user_2|2026-06-01 22:00:00|
|   1179|      2026-06-16| South|       Furniture|    2301.12|Mumbai| 16|     Premium|1411.37|      11|             NULL|  user_3|2026-06-16 08:00:00|
|   1055|      2026-02-09| North|     Electronics|     554.45|Jaipur| 38|       Basic|1227.27|

# Q11: Explain Shuffle and why it is a wide transformation.

# Shuffle is the process of redistributing data across partitions during operations such as groupBy(), join(), and distinct().

It is called a wide transformation because data moves between different partitions and worker nodes.

Shuffle is expensive because it increases network and disk usage.

# Q12: Remove rows where email is null OR username is an empty string.

In [33]:
clean_df = df.filter(
    col("email").isNotNull()
)

clean_df = clean_df.filter(
    col("username") != ""
)

clean_df.show(5)

+-------+----------------+------+----------------+-----------+------+---+------------+-------+--------+-----------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|  city|age|subscription|  price|store_id|            email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+------+---+------------+-------+--------+-----------------+--------+-------------------+
|   1028|      2026-06-13|  West|        Clothing|    1299.97| Delhi| 22|    Standard| 1494.3|      18|user1@example.com|  user_1|2026-06-13 02:00:00|
|   1108|      2026-06-01|  West|     Electronics|     559.11| Delhi| 48|    Standard| 101.75|       7|user2@example.com|  user_2|2026-06-01 22:00:00|
|   1055|      2026-02-09| North|     Electronics|     554.45|Jaipur| 38|       Basic|1227.27|       2|user4@example.com|  user_4|2026-02-09 23:00:00|
|   1137|      2026-04-28|  West|       Groceries|     486.12|Mumbai| 56|    Standard|1776.63|

# Q13: Calculate min, max and mean of the price column using .agg().

In [34]:
df.agg(

    min("price").alias("Minimum Price"),

    max("price").alias("Maximum Price"),

    avg("price").alias("Average Price")

).show()

+-------------+-------------+------------------+
|Minimum Price|Maximum Price|     Average Price|
+-------------+-------------+------------------+
|        50.56|      1999.91|1005.0782857142856|
+-------------+-------------+------------------+



# Q14: What is the risk of using inferSchema=true with inconsistent date formats?

If the date formats are inconsistent, Spark may infer incorrect data types.

Some values may become null or entire columns may be converted to strings.

This can lead to incorrect analysis and poor data quality.

# Q15: Build a final processing pipeline.

In [36]:
final_pipeline = (

    df

    .dropDuplicates()

    .na.fill(
        {"price": 0}
    )

    .groupBy(
        "store_id"
    )

    .agg(
        sum("price")
        .alias("total_revenue")
    )

)

final_pipeline.show()

+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
|      12| 48181.21999999999|
|       1|          33237.85|
|      13| 54141.75999999999|
|      16|           57230.8|
|       6|48278.579999999994|
|       3| 52542.25999999999|
|      20| 50828.74999999999|
|       5|           42890.5|
|      19|49844.860000000015|
|      15|          56255.91|
|      17|          43884.26|
|       9| 42870.23000000001|
|       4|          38542.92|
|       8| 42833.65000000001|
|       7| 46984.09000000001|
|      10| 36917.66999999999|
|      11| 58540.12999999999|
|      14| 56835.83000000001|
|       2|42507.170000000006|
|      18| 62898.45999999999|
+--------+------------------+



In [38]:
# Save Final Processed Dataset

cleaned_df = (

    df

    .dropDuplicates()

    .na.fill(
        {"price": 0}
    )

)

cleaned_df.toPandas().to_csv(

    "../output/cleaned_dataset.csv",

    index=False

)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
